# 01 · Introduction to Quantum Computing

Welcome to **Quantum AI Basics**. This first notebook builds the intuition you need before touching any quantum machine learning. We cover:

1. Why quantum computing matters
2. The **qubit** vs. the classical bit
3. **Superposition** — being in many states at once
4. **Measurement** — collapsing to a classical answer
5. **Entanglement** — the "spooky" correlation that powers quantum advantage

All code uses **Qiskit 1.x** (the modern IBM Quantum SDK) and runs locally on a simulator — no quantum hardware or cloud account required.

## Setup

Install the dependencies (run once). On most machines this takes a minute or two.

In [ ]:
# Run once to install. Comment out after the first run.
%pip install qiskit qiskit-aer matplotlib pylatexenc

## 1. Why quantum computing?

Classical computers store information in **bits** that are strictly `0` or `1`. A quantum computer uses **qubits**, which obey the rules of quantum mechanics and can exist in a *combination* of `0` and `1` simultaneously.

The practical payoff: `n` qubits can represent a superposition of all `2^n` possible bit-strings at once. With 300 qubits you already have more basis states than there are atoms in the observable universe. Quantum algorithms try to exploit this enormous state space — together with **interference** and **entanglement** — to solve certain problems (factoring, simulation of molecules, some optimization and machine-learning tasks) faster than the best known classical methods.

## 2. The qubit

A qubit's state is written as a combination of two **basis states** $|0\rangle$ and $|1\rangle$ (this is *Dirac / bra-ket notation*):

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle$$

- $\alpha$ and $\beta$ are complex numbers called **amplitudes**.
- $|\alpha|^2$ is the probability of measuring `0`, and $|\beta|^2$ the probability of measuring `1`.
- They must satisfy the **normalization** rule $|\alpha|^2 + |\beta|^2 = 1$.

A qubit can be pictured as an arrow on the surface of the **Bloch sphere**: $|0\rangle$ at the north pole, $|1\rangle$ at the south pole, and every superposition somewhere in between.

In [ ]:
from qiskit import QuantumCircuit

# A circuit with one qubit. By default it starts in state |0>.
qc = QuantumCircuit(1)
qc.draw("mpl")

## 3. Superposition with the Hadamard gate

The **Hadamard gate** (`H`) takes a definite $|0\rangle$ and puts it into an *equal* superposition:

$$H|0\rangle = \tfrac{1}{\sqrt{2}}\big(|0\rangle + |1\rangle\big)$$

Now there is a 50% chance of measuring `0` and a 50% chance of measuring `1`. This is the quantum equivalent of a perfectly fair coin — but, crucially, *before* we look the qubit is genuinely in both states at once.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)            # put the qubit into superposition
qc.measure_all()   # measure all qubits into a classical register
qc.draw("mpl")

## 4. Measurement

Measurement **collapses** the superposition to a single classical value, with probabilities given by the squared amplitudes. We can't read out $\alpha$ and $\beta$ directly — we only ever see `0` or `1`.

To estimate the underlying probabilities we run the circuit many times ("**shots**") and count outcomes. We use Qiskit's modern **`StatevectorSampler`** primitive, which simulates sampling from the circuit's output distribution.

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1024)   # 1024 repetitions
result = job.result()

counts = result[0].data.meas.get_counts()
print("Measurement counts:", counts)

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts)

You should see roughly **~512 / ~512** — close to 50/50, with small statistical fluctuations. Increase `shots` and the split gets closer to perfectly even.

## 4b. Going deeper: coherence, decoherence, and what "measurement" really means

The superposition above is **fragile**. Two ideas make this precise:

**Coherence.** A qubit is in a *coherent* superposition only as long as its amplitudes **and their relative phase** are undisturbed by the outside world. That phase is exactly what makes quantum interference possible — lose it and the qubit degrades into an ordinary random bit.

**Decoherence.** Any stray interaction — heat, vibration, a stray field, a passing photon — leaks information about the qubit into the environment and **scrambles the phase**, quietly collapsing the superposition into a classical mixture. This is *the* central engineering problem of quantum computing, and it's why real chips are heavily shielded and cooled to near absolute zero.

Hardware engineers track two timescales:

- **T₁ (relaxation time)** — how long before the qubit loses energy and decays from $|1\rangle$ toward $|0\rangle$.
- **T₂ (dephasing time)** — how long the delicate *phase* between $|0\rangle$ and $|1\rangle$ survives. T₂ is usually the stricter limit, and every gate in your circuit must finish well within it.

**And "measurement" is not mystical.** It does *not* require a conscious observer. *Any* physical interaction that records the qubit's state — a photon striking a detector, an atom colliding with another — counts as a measurement and collapses the superposition. Decoherence is, in effect, the environment "measuring" your qubit without permission.

## 5. Entanglement — the Bell state

**Entanglement** links qubits so that the state of one is correlated with another, no matter the distance between them. The simplest example is a **Bell state**, made with a Hadamard followed by a **CNOT** (controlled-NOT) gate:

$$|\Phi^+\rangle = \tfrac{1}{\sqrt{2}}\big(|00\rangle + |11\rangle\big)$$

When we measure, we *only ever* get `00` or `11` — never `01` or `10`. Measuring one qubit instantly tells you the other, even though each individual qubit looked random. This correlation has no classical analogue and is a key resource for quantum algorithms.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)          # superposition on qubit 0
bell.cx(0, 1)      # entangle qubit 0 (control) with qubit 1 (target)
bell.measure_all()
bell.draw("mpl")

In [ ]:
job = sampler.run([bell], shots=1024)
counts = job.result()[0].data.meas.get_counts()
print("Bell-state counts:", counts)
plot_histogram(counts)

Notice only `00` and `11` appear — the qubits are perfectly correlated.

## 5b. Going deeper: Einstein's "spooky action", Bell's theorem, and what entanglement is *for*

Entanglement bothered Einstein. In the **1935 EPR paper**, he, Podolsky and Rosen argued that if measuring one particle instantly fixes its distant partner, then quantum mechanics must be *incomplete* — surely the answer was "hidden" inside the particles all along (**local hidden variables**). Einstein called the effect **"spooky action at a distance."**

In **1964 John Bell** turned this philosophical dispute into an *experiment*. His **Bell inequality** is a limit that any local-hidden-variable theory must obey; quantum mechanics predicts it can be **violated**. Decades of ever-tighter experiments — culminating in **loophole-free Bell tests since 2015** — have confirmed the violation to better than **8 standard deviations**, ruling out local realism. (This line of work won the **2022 Nobel Prize in Physics**.) Nature really is "spooky"; the correlations are genuinely real.

One crucial subtlety: entanglement does **not** let you send information faster than light. Each side alone sees only random outcomes — the correlation reveals itself *only* when the two parties later compare notes over an ordinary classical channel.

By 2026, entanglement is no longer a curiosity but a working **engineering resource**:

- **Quantum teleportation** — transfer an *unknown* qubit state from Alice to Bob by consuming a shared entangled (Bell) pair plus **two classical bits**. The qubit itself never travels the channel.
- **Quantum key distribution (QKD)** — share encryption keys whose secrecy is guaranteed by physics: any eavesdropper unavoidably disturbs the quantum state and is detected.
- **Quantum networks** — entanglement distributed over distance is the backbone of the emerging "quantum internet." Real links already exist (e.g. China's **2,000 km** Beijing–Shanghai network; in late 2025 India demonstrated a QKD network over **500+ km**).
- **Error correction** — the logical qubits on every fault-tolerant roadmap (notebook 05) are built from *entangled* groups of physical qubits.

So the Bell state you just built isn't a toy — it's the same primitive behind teleportation, quantum-secure communication, and quantum computing itself.

## Summary

| Concept | One-line takeaway |
|---|---|
| **Qubit** | A two-level quantum system; state = $\alpha|0\rangle + \beta|1\rangle$ |
| **Superposition** | A qubit can be `0` and `1` at the same time (created with `H`) |
| **Measurement** | Collapses the state to `0`/`1` with probability $|\alpha|^2$ / $|\beta|^2$ |
| **Entanglement** | Correlated qubits (e.g. the Bell state) with no classical analogue |

**Next:** `02_Quantum_Gates_and_Circuits.ipynb` — the full gate set and how to assemble circuits.